# UK House Price Analysis

This notebook explores UK housing prices data using Python. The goal is to clean the dataset, verify sales volume consistency, and analyse regional housing trends.

Dataset source: UK House Price Index.

Tools used: pandas, numpy, seaborn.

In [11]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=DeprecationWarning)

import numpy as np
import pandas as pd
import seaborn as sns

df = pd.read_csv("_datasets/UK_houseprices2026.csv")
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58320 entries, 0 to 58319
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Date                   58320 non-null  object 
 1   Region                 58320 non-null  object 
 2   Mean Price             58320 non-null  int64  
 3   Sales Volume           57510 non-null  float64
 4   New Price              55948 non-null  float64
 5   New Sales Volume       53484 non-null  float64
 6   Old Price              55948 non-null  float64
 7   Old Sales Volume       55940 non-null  float64
 8   Detached Price         56592 non-null  float64
 9   Semi-Detached Price    56592 non-null  float64
 10  Terraced Price         56703 non-null  float64
 11  Flat Price             56736 non-null  float64
 12  Cash Price             56448 non-null  float64
 13  Cash Sales Volume      55660 non-null  float64
 14  Mortgage Price         56448 non-null  float64
 15  Mo

,Mean Price,Sales Volume,New Price,New Sales Volume,Old Price,Old Sales Volume,Detached Price,Semi-Detached Price,Terraced Price,Flat Price,Cash Price,Cash Sales Volume,Mortgage Price,Mortgage Sales Volume,FTB Price,FOO Price
count,5.832000e+04,57510.000000,5.594800e+04,53484.000000,5.594800e+04,55940.000000,5.659200e+04,5.659200e+04,5.670300e+04,5.673600e+04,5.644800e+04,55660.000000,5.644800e+04,55662.000000,5.644800e+04,5.644800e+04
mean,2.627412e+05,1272.902886,3.122324e+05,141.876823,2.622279e+05,1163.125277,4.908798e+05,3.036792e+05,2.457103e+05,1.653566e+05,2.651402e+05,323.518847,2.669992e+05,746.814559,2.226672e+05,3.196021e+05
std,1.500519e+05,7812.329002,1.425918e+05,884.398255,1.514805e+05,7075.118949,4.418184e+05,2.875523e+05,2.478339e+05,1.196029e+05,1.539273e+05,2009.839273,1.502800e+05,4612.950860,1.261562e+05,1.840619e+05
min,6.603700e+04,3.000000,7.027900e+04,1.000000,6.572300e+04,3.000000,1.028740e+05,6.834100e+04,5.311900e+04,4.095400e+04,6.269300e+04,1.000000,6.688600e+04,1.000000,6.142800e+04,7.260400e+04
25%,1.596375e+05,136.000000,2.156055e+05,9.000000,1.593048e+05,123.000000,2.667325e+05,1.652540e+05,1.310205e+05,9.679600e+04,1.596815e+05,37.000000,1.648805e+05,88.000000,1.403952e+05,1.927710e+05
50%,2.230535e+05,199.000000,2.825615e+05,20.000000,2.227585e+05,179.000000,3.691345e+05,2.294675e+05,1.842560e+05,1.322715e+05,2.258415e+05,58.000000,2.272855e+05,136.000000,1.916680e+05,2.685745e+05
75%,3.213855e+05,335.000000,3.753618e+05,44.000000,3.202295e+05,306.000000,5.519000e+05,3.414255e+05,2.773355e+05,1.908755e+05,3.241085e+05,102.000000,3.252920e+05,238.000000,2.655032e+05,3.905145e+05
max,1.656986e+06,183609.000000,1.414204e+06,17812.000000,1.665089e+06,166098.000000,5.820174e+06,3.919683e+06,3.888915e+06,1.330583e+06,1.621751e+06,52162.000000,1.686923e+06,128426.000000,1.417178e+06,1.956287e+06


## Data Preparation

We extract the relevant columns for sales data and remove missing values from the mean price column.

In [2]:
sales = df.iloc[:, :8]
avgPrice = sales["Mean Price"].dropna()
sales.head()

,Date,Region,Mean Price,Sales Volume,New Price,New Sales Volume,Old Price,Old Sales Volume
0,2014-01-01,Aberdeenshire,192239,411.0,260142.0,62.0,184548.0,349.0
1,2014-02-01,Aberdeenshire,191558,347.0,259718.0,84.0,183857.0,263.0
2,2014-03-01,Aberdeenshire,187155,383.0,253922.0,86.0,179592.0,297.0
3,2014-04-01,Aberdeenshire,189000,513.0,256803.0,100.0,181342.0,413.0
4,2014-05-01,Aberdeenshire,192885,585.0,263044.0,103.0,185020.0,482.0


## Data Consistency Check
In this step we verify the consistency of the dataset.

The total **Sales Volume** should equal the sum of **New Sales Volume** and **Old Sales Volume**.  
We compute the absolute difference and identify rows where the mismatch exceeds a tolerance of 0.5.

This helps detect potential inconsistencies in the dataset.

In [3]:
diff = df["Sales Volume"] - (df["New Sales Volume"] + df["Old Sales Volume"])
abs_diff = diff.abs()

mismatch = abs_diff > 0.5
mismatchCount = mismatch.sum()
mismatchRegions = df.loc[mismatch, "Region"].unique()

print("Mismatch rows:", mismatchCount)
print("Regions affected:", mismatchRegions)

Mismatch rows: 75
Regions affected: ['Inner London' 'North Northamptonshire' 'Northern Ireland'
 'United Kingdom' 'West Northamptonshire']


## Data Cleaning
The dataset includes aggregated entries such as **United Kingdom, England, Scotland, Wales, and Northern Ireland**.

For regional analysis, these aggregated observations are removed.  
Rows containing missing values are also dropped to ensure a clean dataset for further analysis.

In [4]:
remove_regions = ["United Kingdom",
                  "England",
                  "Scotland",
                  "Wales",
                  "Northern Ireland"]
filtered_regions = df[~df["Region"].isin(remove_regions)]

sales_clean = filtered_regions.dropna()
sales_clean.head()

,Date,Region,Mean Price,Sales Volume,New Price,New Sales Volume,Old Price,Old Sales Volume,Detached Price,Semi-Detached Price,Terraced Price,Flat Price,Cash Price,Cash Sales Volume,Mortgage Price,Mortgage Sales Volume,FTB Price,FOO Price
0,2014-01-01,Aberdeenshire,192239,411.0,260142.0,62.0,184548.0,349.0,276808.0,168052.0,141761.0,109450.0,179333.0,111.0,199310.0,300.0,150307.0,226869.0
1,2014-02-01,Aberdeenshire,191558,347.0,259718.0,84.0,183857.0,263.0,275641.0,167303.0,141339.0,109561.0,178732.0,79.0,198590.0,268.0,150054.0,225868.0
2,2014-03-01,Aberdeenshire,187155,383.0,253922.0,86.0,179592.0,297.0,270184.0,163266.0,137690.0,106211.0,174452.0,111.0,194095.0,272.0,146308.0,220887.0
3,2014-04-01,Aberdeenshire,189000,513.0,256803.0,100.0,181342.0,413.0,271913.0,165402.0,139679.0,107314.0,176207.0,121.0,195993.0,392.0,148189.0,222767.0
4,2014-05-01,Aberdeenshire,192885,585.0,263044.0,103.0,185020.0,482.0,277360.0,169069.0,142767.0,109078.0,179742.0,140.0,200057.0,445.0,150990.0,227513.0


## Regional Sales Aggregation
We compute the total number of property sales for each region by aggregating the
**Sales Volume** column. This allows us to compare how housing market activity
varies across regions.

In [5]:
soldPerRegion = df.groupby("Region")["Sales Volume"].sum()
soldPerRegion.head()

Region
Aberdeenshire              52333.0
Adur                       11978.0
Amber Valley               26016.0
Angus                      24155.0
Antrim and Newtownabbey    25863.0
Name: Sales Volume, dtype: float64

## Average House Prices in Greater Manchester
We analyse housing price trends in **Greater Manchester** by computing the
average yearly house price based on the *Mean Price* column.

In [6]:
GM = df[df["Region"] == "Greater Manchester"].copy()
GM["Year"] = pd.to_datetime(GM["Date"]).dt.year

annualAvg = GM.groupby("Year")["Mean Price"].mean()
annualAvg

Year
2014    122045.666667
2015    128763.333333
2016    137953.250000
2017    147450.500000
2018    154871.083333
2019    160932.583333
2020    167812.250000
2021    191004.083333
2022    212646.416667
2023    216807.083333
2024    222817.083333
2025    232475.916667
Name: Mean Price, dtype: float64

## Regional Detached House Prices in 2024
We focus on detached house prices in the year **2024**. For each region,
we compute the median detached house price and identify regions whose prices
are above the 95th percentile, representing the most expensive areas.

In [7]:
df_new = df.copy()
df_new["Year"] = pd.to_datetime(df["Date"]).dt.year
df2024 = df_new[df_new["Year"] == 2024]

detached2024 = df2024.groupby("Region")["Detached Price"].median()
expensive = detached2024.quantile(0.95)

detached2024_expensive = detached2024[detached2024 > expensive]
detached2024_expensive

Region
Barnet                    1627087.5
Brent                     1325065.5
Camden                    3462403.0
City of Westminster       4368453.5
Ealing                    1350946.5
Elmbridge                 1504700.0
Enfield                   1204151.0
Hackney                   1324363.0
Hammersmith and Fulham    1829783.0
Haringey                  2220648.0
Harrow                    1180033.0
Inner London              1857506.5
Islington                 1670521.0
Kensington and Chelsea    4760293.0
Kingston upon Thames      1235464.5
Merton                    2198556.5
Richmond upon Thames      1703265.5
Southwark                 1710689.5
St Albans                 1181955.0
Wandsworth                2496095.5
Name: Detached Price, dtype: float64

## Expensive Region Threshold Analysis
We determine the percentile threshold at which **Greater Manchester**
would be classified as an expensive region based on detached house prices.

In [8]:
gm_value = detached2024["Greater Manchester"]
p_gm = (detached2024 <= gm_value).mean()
p_gm

np.float64(0.4024691358024691)

## House Price Index (HPI) Calculation
To analyse long-term housing price trends, we compute a **House Price Index (HPI)**.
The index is normalised so that the value equals **100 in January 2015** for each region.

In [9]:
df_hpi = df.copy()
df_hpi["Date"] = pd.to_datetime(df_hpi["Date"])

base = df_hpi[(df_hpi["Date"].dt.year == 2015) & (df_hpi["Date"].dt.month == 1)]
base_prices = base.set_index("Region")["Mean Price"]

df_hpi["HPI"] = (df_hpi["Mean Price"] / df_hpi["Region"].map(base_prices))*100

df_hpi.head()

,Date,Region,Mean Price,Sales Volume,New Price,New Sales Volume,Old Price,Old Sales Volume,Detached Price,Semi-Detached Price,Terraced Price,Flat Price,Cash Price,Cash Sales Volume,Mortgage Price,Mortgage Sales Volume,FTB Price,FOO Price,HPI
0,2014-01-01,Aberdeenshire,192239,411.0,260142.0,62.0,184548.0,349.0,276808.0,168052.0,141761.0,109450.0,179333.0,111.0,199310.0,300.0,150307.0,226869.0,92.814828
1,2014-02-01,Aberdeenshire,191558,347.0,259718.0,84.0,183857.0,263.0,275641.0,167303.0,141339.0,109561.0,178732.0,79.0,198590.0,268.0,150054.0,225868.0,92.486035
2,2014-03-01,Aberdeenshire,187155,383.0,253922.0,86.0,179592.0,297.0,270184.0,163266.0,137690.0,106211.0,174452.0,111.0,194095.0,272.0,146308.0,220887.0,90.360224
3,2014-04-01,Aberdeenshire,189000,513.0,256803.0,100.0,181342.0,413.0,271913.0,165402.0,139679.0,107314.0,176207.0,121.0,195993.0,392.0,148189.0,222767.0,91.251008
4,2014-05-01,Aberdeenshire,192885,585.0,263044.0,103.0,185020.0,482.0,277360.0,169069.0,142767.0,109078.0,179742.0,140.0,200057.0,445.0,150990.0,227513.0,93.126723


## Cheapest Month to Buy Property
To explore seasonal patterns in housing prices, we determine which month
tends to have the lowest flat prices across regions and years.

In [10]:
df_hpi["Date"] = pd.to_datetime(df_hpi["Date"])
df_hpi["Year"] = df_hpi["Date"].dt.year
df_hpi["Month"] = df_hpi["Date"].dt.month

cheap = df_hpi.groupby(["Region", "Year"])["Flat Price"].idxmin()
cheap = cheap.dropna().astype(int)
cheapest_months = df_hpi["Month"].iloc[cheap.values]
cheapest_month = cheapest_months.value_counts().idxmax()

cheapest_month

np.int32(1)

## Conclusion

This notebook analysed UK house price data using Python.  
The workflow included data cleaning, validation of sales records, regional aggregation,
and exploratory analysis of price trends.

The analysis highlighted regional variation in sales activity and seasonal patterns
in housing prices. A House Price Index (HPI) was also constructed to normalise price
changes over time.